# TechAdmin Database Validation and Analytics
Read-only notebook for connection checks, schema inventory, pandas summaries, data-quality checks, operation-policy checks, and readiness reporting.

## 1. Setup and shared connection

In [16]:
from pathlib import Path
import sys
from typing import Any
import pandas as pd
from IPython.display import display
from sqlalchemy import text

here = Path.cwd().resolve()
project_root = next((p for p in [here, *here.parents] if (p / "App").is_dir()), None)
if project_root is None:
    raise RuntimeError("Start Jupyter from inside the TechAdmin repository")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from App.db.connection import DB_SCHEMA, engine
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
print(f"Project root: {project_root}")
print(f"Configured schema: {DB_SCHEMA}")

Project root: D:\ATechAdmin\TechAdmin
Configured schema: techadmin


## 2. Read-only query helper

In [17]:
def query_df(sql: str, params: dict[str, Any] | None = None) -> pd.DataFrame:
    # Execute a parameterized SELECT statement and return a DataFrame.
    with engine.connect() as connection:
        return pd.read_sql_query(text(sql), connection, params=params or {})

## 3. Connection health

In [18]:
connection_df = query_df("""
SELECT current_database() AS database_name,
       current_user AS database_user,
       current_schema() AS current_schema,
       current_setting('search_path') AS search_path,
       inet_server_addr()::text AS server_address,
       inet_server_port() AS server_port
""")
display(connection_df)

,database_name,database_user,current_schema,search_path,server_address,server_port
0,techadmin_dev,postgres,techadmin,"techadmin,public",::1/128,5432


## 4. Schema, table, and row-count inventory

In [19]:
schemas_df = query_df("""
SELECT schema_name FROM information_schema.schemata
WHERE schema_name NOT LIKE 'pg_%' AND schema_name <> 'information_schema'
ORDER BY schema_name
""")
tables_df = query_df("""
SELECT table_schema, table_name, table_type
FROM information_schema.tables
WHERE table_schema = :schema_name
ORDER BY table_name
""", {"schema_name": DB_SCHEMA})
available_tables = set(tables_df["table_name"].tolist())

display(schemas_df)
display(tables_df)

row_counts = []
for table_name in sorted(available_tables):
    result = query_df(f'SELECT COUNT(*) AS row_count FROM "{DB_SCHEMA}"."{table_name}"')
    row_counts.append({"table_name": table_name, "row_count": int(result.loc[0, "row_count"])})
row_counts_df = pd.DataFrame(row_counts)
display(row_counts_df)

,schema_name
0,public
1,techadmin


,table_schema,table_name,table_type
0,techadmin,app_users,BASE TABLE
1,techadmin,operations,BASE TABLE


,table_name,row_count
0,app_users,7
1,operations,2


## 5. Column inventory
Change `TABLE_NAME` to inspect another discovered table.

In [20]:
TABLE_NAME = "app_users"
if TABLE_NAME not in available_tables:
    raise ValueError(f"Unknown table: {TABLE_NAME}")
columns_df = query_df("""
SELECT ordinal_position, column_name, data_type, is_nullable, column_default
FROM information_schema.columns
WHERE table_schema = :schema_name AND table_name = :table_name
ORDER BY ordinal_position
""", {"schema_name": DB_SCHEMA, "table_name": TABLE_NAME})
display(columns_df)

,ordinal_position,column_name,data_type,is_nullable,column_default
0,1,user_id,uuid,NO,gen_random_uuid()
1,2,entra_object_id,uuid,YES,NaN
2,3,user_principal_name,character varying,NO,NaN
3,4,display_name,character varying,NO,NaN
4,5,department,character varying,YES,NaN
5,6,is_active,boolean,NO,true
6,7,created_at,timestamp with time zone,NO,CURRENT_TIMESTAMP
7,8,updated_at,timestamp with time zone,NO,CURRENT_TIMESTAMP


## 6. Application users and summary

In [21]:
app_users_df = query_df(f"""
SELECT user_id, entra_object_id, user_principal_name, display_name,
       department, is_active, created_at, updated_at
FROM "{DB_SCHEMA}"."app_users"
ORDER BY display_name
""")
display(app_users_df)

user_summary_df = pd.DataFrame({
    "metric": ["Registered users", "Active users", "Inactive users", "Departments represented"],
    "value": [
        len(app_users_df),
        int(app_users_df["is_active"].fillna(False).sum()),
        int((~app_users_df["is_active"].fillna(False)).sum()),
        int(app_users_df["department"].dropna().nunique()),
    ],
})
display(user_summary_df)

department_summary_df = (
    app_users_df.assign(department=app_users_df["department"].fillna("UNSPECIFIED"))
    .groupby("department", as_index=False)
    .agg(total_users=("user_id", "count"), active_users=("is_active", "sum"))
)
display(department_summary_df)

,user_id,entra_object_id,user_principal_name,display_name,department,is_active,created_at,updated_at
0,ef242f27-2b98-43e0-a14d-deacbaaf51e8,ae137d62-060e-4598-9616-60eac23ebb1c,aman.14.gupta@coforge.com,Aman Gupta - GNoida,IT,True,2026-09-09 09:48:23.965802+00:00,2026-09-09 09:48:23.965802+00:00
1,7e37ee55-9d5b-4b9e-b01d-0f26f5dff118,94dde518-4345-4665-a8d3-860c852e669d,aman.3.mishra@coforge.com,Aman Mishra,IT,True,2026-09-09 09:48:23.908187+00:00,2026-09-09 09:48:23.908187+00:00
2,5ff2fa29-297b-415d-a363-c7dce26e5168,bffa5400-3970-45f1-b022-13c39d0c0bd7,amit.bhagat@coforge.com,Amit Bhagat,IT,True,2026-09-09 09:48:23.943905+00:00,2026-09-09 09:48:23.943905+00:00
3,f1964853-bede-44a8-846d-857356ad28fe,d7aac575-8f3e-4fd3-a821-db94e4afcb55,jitender.chauhan@coforge.com,Jitender Chauhan - Global IT,IT,True,2026-09-09 10:22:25.036313+00:00,2026-09-09 10:22:25.036325+00:00
4,897139ee-531c-41f9-99ba-ab552e5e36bf,09c5f133-4294-4ef7-b1df-e958fe339f32,roshan.sah@coforge.com,Roshan Sah,IT,True,2026-09-09 09:48:23.952328+00:00,2026-09-09 09:48:23.952328+00:00
5,1375dc37-9516-4255-b450-993a29c1fcfd,ff614b3d-1fdb-453d-b3d9-809ec9bc26fb,shreesanyog.rath@coforge.com,Shreesanyog Rath,IT,True,2026-09-09 09:48:23.959351+00:00,2026-09-09 09:48:23.959351+00:00
6,842f669d-5bd3-427f-be4f-b6aeb35dd560,None,techadmintestuser@coforge.com,TechAdminTestUser,IT,True,2026-09-16 10:21:37.135335+00:00,2026-09-16 10:21:37.135340+00:00


,metric,value
0,Registered users,7
1,Active users,7
2,Inactive users,0
3,Departments represented,1


,department,total_users,active_users
0,IT,7,7


## 7. User data-quality checks

In [22]:
required_users = ["user_id", "entra_object_id", "user_principal_name", "display_name", "is_active", "created_at", "updated_at"]
user_quality_df = pd.DataFrame({
    "check": ["Missing required values", "Duplicate user_id", "Duplicate Entra ID", "Duplicate UPN ignoring case", "Blank UPN", "Blank display name", "Invalid timestamp order"],
    "issue_count": [
        int(app_users_df[required_users].isna().sum().sum()),
        int(app_users_df["user_id"].duplicated().sum()),
        int(app_users_df["entra_object_id"].duplicated().sum()),
        int(app_users_df["user_principal_name"].astype(str).str.strip().str.lower().duplicated().sum()),
        int(app_users_df["user_principal_name"].fillna("").str.strip().eq("").sum()),
        int(app_users_df["display_name"].fillna("").str.strip().eq("").sum()),
        int((pd.to_datetime(app_users_df["updated_at"], utc=True) < pd.to_datetime(app_users_df["created_at"], utc=True)).sum()),
    ],
})
user_quality_df["status"] = user_quality_df["issue_count"].map(lambda n: "PASS" if n == 0 else "REVIEW")
display(user_quality_df)

,check,issue_count,status
0,Missing required values,1,REVIEW
1,Duplicate user_id,0,PASS
2,Duplicate Entra ID,0,PASS
3,Duplicate UPN ignoring case,0,PASS
4,Blank UPN,0,PASS
5,Blank display name,0,PASS
6,Invalid timestamp order,0,PASS


## 8. Operations catalog and summary

In [23]:
operations_df = query_df(f"""
SELECT operation_id, operation_code, operation_name, execution_type,
       tool_name, script_name, risk_level, requires_approval, is_active, created_at
FROM "{DB_SCHEMA}"."operations"
ORDER BY operation_code
""")
display(operations_df)

operation_summary_df = pd.DataFrame({
    "metric": ["Registered operations", "Active operations", "Inactive operations", "Operations requiring approval"],
    "value": [
        len(operations_df),
        int(operations_df["is_active"].fillna(False).sum()),
        int((~operations_df["is_active"].fillna(False)).sum()),
        int(operations_df["requires_approval"].fillna(False).sum()),
    ],
})
display(operation_summary_df)
display(operations_df.groupby("execution_type", as_index=False).agg(total_operations=("operation_id", "count")))
display(operations_df.groupby("risk_level", as_index=False).agg(total_operations=("operation_id", "count"), approval_required=("requires_approval", "sum")))

,operation_id,operation_code,operation_name,execution_type,tool_name,script_name,risk_level,requires_approval,is_active,created_at
0,f97d0386-a278-4a37-86e0-f39e237b0749,GET_USER_DETAILS,Get User Details,API,GET_USER_DETAILS,None,LOW,False,True,2026-09-09 11:12:35.226988+00:00
1,35afd346-f3eb-4189-b4a3-b8bc2d8b5bec,RESET_PASSWORD,Reset Password,API,RESET_PASSWORD,None,HIGH,True,True,2026-09-09 11:12:35.235407+00:00


,metric,value
0,Registered operations,2
1,Active operations,2
2,Inactive operations,0
3,Operations requiring approval,1


,execution_type,total_operations
0,API,2


,risk_level,total_operations,approval_required
0,HIGH,1,1
1,LOW,1,0


## 9. Operation configuration and suggested policy checks
The final rule is suggested project guidance until formally approved.

In [24]:
allowed_types = {"API", "SCRIPT", "JOB"}
allowed_risks = {"LOW", "MEDIUM", "HIGH", "CRITICAL"}
required_ops = ["operation_id", "operation_code", "operation_name", "execution_type", "tool_name", "risk_level", "requires_approval", "is_active", "created_at"]
operation_quality_df = pd.DataFrame({
    "check": ["Missing required values", "Duplicate operation_id", "Duplicate operation_code", "Invalid execution type", "Invalid risk level", "SCRIPT missing script_name", "API containing script_name", "Suggested policy: active HIGH/CRITICAL without approval"],
    "issue_count": [
        int(operations_df[required_ops].isna().sum().sum()),
        int(operations_df["operation_id"].duplicated().sum()),
        int(operations_df["operation_code"].str.strip().str.upper().duplicated().sum()),
        int((~operations_df["execution_type"].isin(allowed_types)).sum()),
        int((~operations_df["risk_level"].isin(allowed_risks)).sum()),
        int((operations_df["execution_type"].eq("SCRIPT") & operations_df["script_name"].fillna("").str.strip().eq("")).sum()),
        int((operations_df["execution_type"].eq("API") & operations_df["script_name"].fillna("").str.strip().ne("")).sum()),
        int((operations_df["is_active"].eq(True) & operations_df["risk_level"].isin(["HIGH", "CRITICAL"]) & operations_df["requires_approval"].eq(False)).sum()),
    ],
})
operation_quality_df["status"] = operation_quality_df["issue_count"].map(lambda n: "PASS" if n == 0 else "REVIEW")
display(operation_quality_df)

,check,issue_count,status
0,Missing required values,0,PASS
1,Duplicate operation_id,0,PASS
2,Duplicate operation_code,0,PASS
3,Invalid execution type,0,PASS
4,Invalid risk level,0,PASS
5,SCRIPT missing script_name,0,PASS
6,API containing script_name,0,PASS
7,Suggested policy: active HIGH/CRITICAL without approval,0,PASS


## 10. Useful operation filters

In [25]:
print("Active operations")
display(operations_df[operations_df["is_active"].eq(True)])
print("High or critical risk operations")
display(operations_df[operations_df["risk_level"].isin(["HIGH", "CRITICAL"])])
print("Operations requiring approval")
display(operations_df[operations_df["requires_approval"].eq(True)])

Active operations


,operation_id,operation_code,operation_name,execution_type,tool_name,script_name,risk_level,requires_approval,is_active,created_at
0,f97d0386-a278-4a37-86e0-f39e237b0749,GET_USER_DETAILS,Get User Details,API,GET_USER_DETAILS,None,LOW,False,True,2026-09-09 11:12:35.226988+00:00
1,35afd346-f3eb-4189-b4a3-b8bc2d8b5bec,RESET_PASSWORD,Reset Password,API,RESET_PASSWORD,None,HIGH,True,True,2026-09-09 11:12:35.235407+00:00


High or critical risk operations


,operation_id,operation_code,operation_name,execution_type,tool_name,script_name,risk_level,requires_approval,is_active,created_at
1,35afd346-f3eb-4189-b4a3-b8bc2d8b5bec,RESET_PASSWORD,Reset Password,API,RESET_PASSWORD,None,HIGH,True,True,2026-09-09 11:12:35.235407+00:00


Operations requiring approval


,operation_id,operation_code,operation_name,execution_type,tool_name,script_name,risk_level,requires_approval,is_active,created_at
1,35afd346-f3eb-4189-b4a3-b8bc2d8b5bec,RESET_PASSWORD,Reset Password,API,RESET_PASSWORD,None,HIGH,True,True,2026-09-09 11:12:35.235407+00:00


## 11. TechAdmin readiness summary
Use this as the manager-friendly overview.

In [26]:
connection_ok = not connection_df.empty and connection_df.loc[0, "database_name"] == "techadmin_dev" and connection_df.loc[0, "current_schema"] == DB_SCHEMA
schema_ok = DB_SCHEMA in schemas_df["schema_name"].tolist()
readiness_df = pd.DataFrame({
    "area": ["Database connection", "Configured schema", "app_users table", "operations table", "User data quality", "Operation configuration"],
    "status": [
        "PASS" if connection_ok else "REVIEW",
        "PASS" if schema_ok else "REVIEW",
        "PASS" if "app_users" in available_tables else "REVIEW",
        "PASS" if "operations" in available_tables else "REVIEW",
        "PASS" if user_quality_df["issue_count"].sum() == 0 else "REVIEW",
        "PASS" if operation_quality_df["issue_count"].sum() == 0 else "REVIEW",
    ],
    "detail": [
        f"Database={connection_df.loc[0, 'database_name']}", f"Schema={DB_SCHEMA}",
        f"Rows={len(app_users_df)}", f"Rows={len(operations_df)}",
        f"Issues={int(user_quality_df['issue_count'].sum())}",
        f"Issues={int(operation_quality_df['issue_count'].sum())}",
    ],
})
display(readiness_df)

,area,status,detail
0,Database connection,PASS,Database=techadmin_dev
1,Configured schema,PASS,Schema=techadmin
2,app_users table,PASS,Rows=7
3,operations table,PASS,Rows=2
4,User data quality,REVIEW,Issues=1
5,Operation configuration,PASS,Issues=0


## 12. Safe query explorer and exact-UPN search

In [27]:
def read_table(table_name: str, limit: int = 100) -> pd.DataFrame:
    # Read only a discovered table and restrict output size.
    if table_name not in available_tables:
        raise ValueError(f"Available tables: {sorted(available_tables)}")
    if not isinstance(limit, int) or not 1 <= limit <= 1000:
        raise ValueError("limit must be between 1 and 1000")
    return query_df(f'SELECT * FROM "{DB_SCHEMA}"."{table_name}" LIMIT {limit}')

display(read_table("operations", 20))

SEARCH_UPN = "aman.3.mishra@coforge.com"
user_search_df = query_df(f"""
SELECT user_id, entra_object_id, user_principal_name, display_name,
       department, is_active, created_at, updated_at
FROM "{DB_SCHEMA}"."app_users"
WHERE LOWER(user_principal_name) = LOWER(:upn)
""", {"upn": SEARCH_UPN.strip()})
display(user_search_df)

,operation_id,operation_code,operation_name,execution_type,tool_name,script_name,risk_level,requires_approval,is_active,created_at
0,f97d0386-a278-4a37-86e0-f39e237b0749,GET_USER_DETAILS,Get User Details,API,GET_USER_DETAILS,None,LOW,False,True,2026-09-09 11:12:35.226988+00:00
1,35afd346-f3eb-4189-b4a3-b8bc2d8b5bec,RESET_PASSWORD,Reset Password,API,RESET_PASSWORD,None,HIGH,True,True,2026-09-09 11:12:35.235407+00:00


,user_id,entra_object_id,user_principal_name,display_name,department,is_active,created_at,updated_at
0,7e37ee55-9d5b-4b9e-b01d-0f26f5dff118,94dde518-4345-4665-a8d3-860c852e669d,aman.3.mishra@coforge.com,Aman Mishra,IT,True,2026-09-09 09:48:23.908187+00:00,2026-09-09 09:48:23.908187+00:00


## 13. Future analytics readiness

In [28]:
future_capabilities = {
    "operation_requests": "Request lifecycle and source-channel analytics",
    "operation_executions": "Success, failure, duration, error, and retry analytics",
    "approval_requests": "Approval status, expiry, and turnaround analytics",
}
future_readiness_df = pd.DataFrame([
    {"table_name": name, "available": name in available_tables, "future_capability": capability}
    for name, capability in future_capabilities.items()
])
display(future_readiness_df)

,table_name,available,future_capability
0,operation_requests,False,Request lifecycle and source-channel analytics
1,operation_executions,False,"Success, failure, duration, error, and retry analytics"
2,approval_requests,False,"Approval status, expiry, and turnaround analytics"


In [30]:
# all tables in our techadmin db

tables = pd.read_sql("SELECT table_name FROM information_schema.tables WHERE table_schema = 'techadmin'", con=engine)
tables

,table_name
0,operation_requests
1,app_users
2,operations


In [31]:
# show each table attributes

for table in tables['table_name']:
    print(f"Attributes for table '{table}':")
    attributes = pd.read_sql(f"SELECT column_name, data_type, is_nullable FROM information_schema.columns WHERE table_schema = 'techadmin' AND table_name = '{table}'", con=engine)
    print(attributes)
    print("\n")

Attributes for table 'operation_requests':
           column_name                 data_type is_nullable
0           request_id                      uuid          NO
1         requested_by                      uuid          NO
2         operation_id                      uuid          NO
3       source_channel         character varying          NO
4     original_request                      text          NO
5          target_type         character varying          NO
6     target_reference         character varying          NO
7     target_object_id                      uuid         YES
8   request_parameters                     jsonb          NO
9    intent_confidence                   numeric         YES
10              status         character varying          NO
11        requested_at  timestamp with time zone          NO
12        completed_at  timestamp with time zone         YES


Attributes for table 'app_users':
           column_name                 data_type is_nullable
0     

# How to use and leverage the notebook

## During development
1. Run all cells after model, seed, or migration changes.
2. Confirm the readiness summary shows `PASS`.
3. Investigate quality checks marked `REVIEW`.
4. Use the table explorer and UPN search for troubleshooting.
5. Validate active, high-risk, and approval-controlled operations.

## Before a demo or update
1. Run the connection, inventory, summary, quality, and readiness sections.
2. Present the readiness table as the top-level status.
3. Use calculated summaries instead of manually inspecting PostgreSQL.
4. Clear outputs before sharing.

## Before and after migrations
1. Run the notebook before migration.
2. Apply the migration.
3. Rerun table inventory, row counts, and quality checks.
4. Investigate unexpected changes.

## Safety and Git hygiene
- Keep the notebook read-only.
- Never add passwords, tokens, or secrets.
- Do not commit outputs containing UPNs, Entra IDs, or server details.
- Before committing, use **Restart Kernel and Clear All Outputs**, then save.
- Keep `.env` ignored.